<a href="https://colab.research.google.com/github/cqx931/AsWeMaySpeak/blob/main/week6/Fine_tuning_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Fine-tuning LLMs

Training LLMs consumes a lot of computational power and energy and so as fine-tuning them. We are using Google Colab for this notebook, so that you can still run some example code without the best specs on your own laptop. Of course you can also run it on your local machine, it's just going to take more time probably.

We are going to make use of the free tier of Google Colab. Change your runtime type by going to the lower right corner of your screen. Select 'T4 GPU' for hardware, it's going to make everything faster.

First let's install the libraries we need.



In [ ]:
! pip install transformers datasets torch

#### Step1: Prepare the Dataset
We are going to use a joke dataset from [here](https://github.com/taivop/joke-dataset). We are using this dataset because it contains short interesting sentences. For longer text, you will need to provide way more training data to get some meaningful result. LLM needs **a lot of** data even for fine tuning. You are not going to get good generative result if you just feed one txt file for similar style of writing. Also GPT-2 is only trained on English so it won't work with other languages.

Let's first download the original dataset from the source. It's going to show up in the folder icon on the left of your Google Colab.

In [ ]:
! curl -O https://raw.githubusercontent.com/taivop/joke-dataset/master/reddit_jokes.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 65.4M  100 65.4M    0     0  26.6M      0  0:00:02  0:00:02 --:--:-- 26.6M


We will only use a subset of it as our training data so that we don't need to wait for hours during the class. Feel free to train through it properly on your own if you are interested in doing so. The following code is going to generate a new `input-text` file in the folder.

We are also doing some preprocessing here so that the text file matches the input format for gpt-2.

In [ ]:
import json
from pathlib import Path

jokes_raw = json.loads(Path("reddit_jokes.json").read_text())

# only get 5000 jokes for quick demo, filter out the short jokes
jokes_parsed = "<|endoftext|>".join("{0}|{1}".format(j['title'], j['body']) for j in jokes_raw[:5000] if len(j['body']) < 50)

# processed data is saved as input-text.txt
Path("input-text.txt").write_text(jokes_parsed)


249127

Now we can create a dataset with the new txt file.

In [ ]:
from datasets import load_dataset

# Create a Dataset object
dataset = load_dataset('text', data_files={'train': 'input-text.txt'})

Generating train split: 0 examples [00:00, ? examples/s]

Each model has its own dictionary of text to token mapping, which means that you have to use the correct tokenizer or the LLM you're trying to train will have no idea what the text you've inputted means.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments


In [ ]:

def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128  # adjust based on your average sentence size
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token # GPT-2 has no pad token by default

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

---
#### Step2: Train and Save
Now we are ready to train. Let's begin by loading the pretrained GPT-2 model.

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

We are only training 10 epochs here to save some time. You can train it over more epochs and observe the trend of Training Loss. Generally Training Loss shall decrease over time, if it gets consistently higher after some point, your model can be overtained. Then it's smart to stop training before this point.

We are also saving the finetuned model, so that we can acess it later.

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    logging_steps=50,
    save_steps=100,
    save_total_limit=2,
    logging_dir='./logs'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
)

trainer.train()

# Save the trained model
trainer.save_model("gpt2-finetuned")
tokenizer.save_pretrained("gpt2-finetuned")

print("Model trained and saved as gpt2-finetuned")
# Training Time on Google Colab: about 4 min

Step,Training Loss
50,2.649700


KeyboardInterrupt: 

---
#### Step 3: Load and Test

In [ ]:
# folder where you saved your model
model_path = "gpt2-finetuned"

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

# Important for GPT-2 to get correct results.
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_path)

print("Tokenizer vocab:", len(tokenizer))
print("Model vocab:", model.config.vocab_size)

Tokenizer vocab: 50257
Model vocab: 50257


We can check our fine-tuned model using the following function

In [ ]:
import torch

def take_prompt_output_response(prompt):
 # Tokenize the input prompt
  input_ids = tokenizer.encode(prompt, return_tensors='pt')

  # Create attention mask (1 for real tokens, 0 for padding tokens)
  attention_mask = torch.ones(input_ids.shape, dtype=torch.long)

  # Generate text
  output = model.generate(
      input_ids,
      attention_mask=attention_mask,
      max_length=100,  # Adjust the max length to control the output length
      num_return_sequences=1,
      no_repeat_ngram_size=2,
      top_k=50,
      top_p=0.95,
      temperature=0.7, # feel free to tweak this parameter to get wilder result
      do_sample=True,
      pad_token_id=tokenizer.eos_token_id  # Explicitly set pad_token_id to eos_token_id
  )

  # Decode and print the generated text
  generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
  print(generated_text)

# Call the model with the prompt
take_prompt_output_response("A guy walks into a bar")


A guy walks into a bar and says "I want to have a glass of iced coffee"|*sigh*


---
#### Projects using GPT-2/GPT-3
- [Digital Folktales](https://www.fabianmosele.com/digital-folktales) by [Fabian Mosele](https://www.fabianmosele.com/), 2022, GPT-3
- [The Infinite Conversation](https://jamez.it/project/the-infinite-conversation/), by [Giacomo Miceli](https://jamez.it/), 2022, GPT-3
- [Posthuman Monsters](https://www.posthumanmonsters.net/) by [Guida Ribeiro](https://guidaribeiro.net/), 2021, GPT-2
- ...

##### References
- [gpt2-jokes](https://pmbaumgartner.github.io/blog/gpt2-jokes/)
- [fine-tune-gpt2](https://colab.research.google.com/drive/1iv728ixGl7vNEWUugSR9OoJ0jG2RYuix?usp=sharing#scrollTo=uylQJccvtzDA)